In [1]:
from datasets import Dataset, DatasetDict
from setfit import SetFitModel, Trainer, TrainingArguments, sample_dataset
import pandas as pd
import mlflow
from mlflow.pyfunc import PythonModel
mlflow.transformers.autolog()

In [2]:
mlflow.set_experiment("adeptID")

<Experiment: artifact_location='file:///c:/Users/jvhua/OneDrive/Desktop/ISYE-CSE-MGT-6748-Group-1/preprocess/mlruns/936985963919367918', creation_time=1718591276177, experiment_id='936985963919367918', last_update_time=1718591276177, lifecycle_stage='active', name='adeptID', tags={}>

In [3]:
df = pd.read_excel(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\set-fit-datav2.xlsx")
df.columns
df = df[['id', 'text', 'label']].reset_index(drop = True)

In [4]:
dataset = Dataset.from_pandas(df)

train_val_test_split = dataset.train_test_split(test_size=0.3)

train_val_split = train_val_test_split['train'].train_test_split(test_size=0.5)

dataset_dict = DatasetDict({
    'train': train_val_split['train'],
    'validation': train_val_split['test'],
    'test': train_val_test_split['test']
})

In [5]:
# dataset_dict.save_to_disk(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\lightcast_chunks_3500_v3.hf")
dataset_dict


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1224
    })
    validation: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1225
    })
    test: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1050
    })
})

In [6]:
train_dataset = dataset_dict['train']
eval_dataset = dataset_dict["validation"]
test_dataset = dataset_dict["test"]

In [10]:
# Load a SetFit model from Hub
model = SetFitModel.from_pretrained(
    "nomic-ai/nomic-embed-text-v1.5", 
    labels=[0, 1],
    trust_remote_code=True,
)

<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
<All keys matched successfully>
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [11]:
args = TrainingArguments(
    batch_size=8,
    num_epochs=5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    seed = 7,
    warmup_proportion = 0.1,
    num_iterations=5
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric="accuracy",
    column_mapping={"text": "text", "label": "label"}
)

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

In [12]:
with mlflow.start_run(run_name = "setfit-preprocess-model-train"):
    trainer.train()
    metrics = trainer.evaluate(test_dataset)
    mlflow.log_metric('accuracy',  metrics['accuracy'])
    print(metrics)

***** Running training *****
  Num unique pairs = 12240
  Batch size = 8
  Num epochs = 5
  Total optimization steps = 7650


  0%|          | 0/7650 [00:00<?, ?it/s]

  0%|          | 0/7650 [00:00<?, ?it/s]

{'embedding_loss': 0.2533, 'learning_rate': 2.6143790849673203e-08, 'epoch': 0.0}
{'embedding_loss': 0.2218, 'learning_rate': 1.3071895424836604e-06, 'epoch': 0.03}
{'embedding_loss': 0.256, 'learning_rate': 2.6143790849673208e-06, 'epoch': 0.07}
{'embedding_loss': 0.2973, 'learning_rate': 3.92156862745098e-06, 'epoch': 0.1}
{'embedding_loss': 0.2479, 'learning_rate': 5.2287581699346416e-06, 'epoch': 0.13}
{'embedding_loss': 0.2888, 'learning_rate': 6.535947712418301e-06, 'epoch': 0.16}
{'embedding_loss': 0.24, 'learning_rate': 7.84313725490196e-06, 'epoch': 0.2}
{'embedding_loss': 0.225, 'learning_rate': 9.150326797385621e-06, 'epoch': 0.23}
{'embedding_loss': 0.1449, 'learning_rate': 1.0457516339869283e-05, 'epoch': 0.26}
{'embedding_loss': 0.1545, 'learning_rate': 1.1764705882352942e-05, 'epoch': 0.29}
{'embedding_loss': 0.2337, 'learning_rate': 1.3071895424836602e-05, 'epoch': 0.33}
{'embedding_loss': 0.0016, 'learning_rate': 1.4379084967320263e-05, 'epoch': 0.36}
{'embedding_loss'

  0%|          | 0/1532 [00:00<?, ?it/s]

...

{'eval_embedding_loss': 0.2631, 'learning_rate': 0.0, 'epoch': 5.0}


Loading best SentenceTransformer model from step 1530.
<All keys matched successfully>


{'train_runtime': 4549.5122, 'train_samples_per_second': 13.452, 'train_steps_per_second': 1.681, 'epoch': 5.0}


Applying column mapping to the evaluation dataset
***** Running evaluation *****


{'accuracy': 0.8580952380952381}


In [16]:
save_directory = r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\setfit_nomic_model"
trainer.model._save_pretrained(save_directory=save_directory)

In [17]:
class SetFitWrapper(PythonModel):

    def load_context(self, context):
        self.model = SetFitModel.from_pretrained(context.artifacts['model'])

    def predict(self, context, model_input):
        return self.model.predict(model_input)

In [18]:
with mlflow.start_run(run_name = "setfit-preprocess-model-train"):
    mlflow.pyfunc.log_model(
        artifact_path='setfit_model',
        python_model=SetFitWrapper(),
        artifacts={'model': save_directory}
    )
mlflow.end_run()

2024/06/19 12:06:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\jvhua\AppData\Local\Temp\tmpzwj0ws5v\model, flavor: python_function). Fall back to return ['cloudpickle==3.0.0']. Set logging level to DEBUG to see the full traceback. 
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\_distutils_hack\__init__.py:11: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing di